# 24 — Validação da verificação orientada ao artigo

Esta etapa valida a resposta ao usuário que envia um artigo. O sistema separa três conceitos: compatibilidade textual, força da literatura recuperada e confiança na cobertura da própria verificação. Nenhum deles representa probabilidade de verdade.

## Critérios de aceite

1. Trechos neutros contam como contexto, não como confirmação.
2. O índice informa cobertura, volume, data, citações e ramificação.
3. Ensaios clínicos aparecem como não aplicáveis para ciência básica.
4. A saída não chama o artigo de verdadeiro ou falso.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / 'src'))

from fatofake.verification_cards import build_verification_confidence

In [2]:
articles = (
    {
        'publication_date': '2024',
        'assessments': [{'relation': 'NEUTRAL'}],
        'retrieval': {
            'sources': ['PubMed relacionados', 'OpenAlex'],
            'citation_count': 25,
            'related_work_count': 10,
        },
    },
    {
        'publication_date': '2020',
        'assessments': [{'relation': 'SUPPORTS'}],
        'retrieval': {
            'sources': ['PubMed'],
            'citation_count': 12,
            'related_work_count': 6,
        },
    },
)

confidence = build_verification_confidence(
    articles=articles, research_context='BASIC_SCIENCE'
)
confidence

{'score': 73,
 'level': 'MODERATE',
 'label': 'confiança da verificação',
 'explanation': 'Mede quanto da checagem foi coberto por dados recuperados. Não é a probabilidade de o artigo estar correto.',
 'components': [{'code': 'ANALYSIS_COVERAGE',
   'label': 'Cobertura dos textos',
   'points': 30.0,
   'maximum': 30,
   'detail': '2 de 2 artigos tiveram abstract analisado.',
   'status': 'AVAILABLE'},
  {'code': 'INDEPENDENT_EVIDENCE',
   'label': 'Volume de evidência independente',
   'points': 4.0,
   'maximum': 20,
   'detail': '1 artigo(s) compararam diretamente a alegação; 1 forneceram apenas contexto relacionado.',
   'status': 'AVAILABLE'},
  {'code': 'RECENCY',
   'label': 'Atualidade das publicações',
   'points': 15.0,
   'maximum': 15,
   'detail': '2 publicação(ões) tinham ano identificável.',
   'status': 'AVAILABLE'},
  {'code': 'CITATIONS',
   'label': 'Contexto de citações',
   'points': 7.9,
   'maximum': 15,
   'detail': '37 citação(ões) registradas em 2 artigo(s).',

In [3]:
volume = next(item for item in confidence['components'] if item['code'] == 'INDEPENDENT_EVIDENCE')
trials = next(item for item in confidence['components'] if item['code'] == 'CLINICAL_TRIALS')

assert '1 artigo(s) compararam diretamente' in volume['detail']
assert '1 forneceram apenas contexto' in volume['detail']
assert trials['status'] == 'NOT_APPLICABLE'
assert 'probabilidade' in confidence['explanation']
assert 'verdadeiro' not in str(confidence).lower()
assert 'falso' not in str(confidence).lower()

print(f"Índice de cobertura da verificação: {confidence['score']}% ({confidence['level']})")
print(volume['detail'])
print(trials['detail'])

Índice de cobertura da verificação: 73% (MODERATE)
1 artigo(s) compararam diretamente a alegação; 1 forneceram apenas contexto relacionado.
Não aplicável ao desenho identificado para o artigo.


## Interpretação

O índice aumenta quando há mais textos efetivamente analisados, publicações recentes, metadados de citação e uma rede bibliográfica observável. Ele não aumenta apenas porque um artigo é muito citado e não transforma conteúdo relacionado em apoio direto. Para estudos básicos, a ausência de ensaios clínicos não reduz a nota, pois esse critério não se aplica ao desenho.